# Notebook 03a: K-Fold Cross-Validation Analysis

After running `python -m src.kfold_cv`, this notebook aggregates the per-fold results and produces:
- Mean +/- std across folds
- ROC and PR curves
- Confusion matrix
- Per-class clinical metrics

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix

## Load per-fold results

In [ ]:
RUN_DIR = Path('../runs/cv_modified_convnext')  # adjust

df = pd.read_csv(RUN_DIR / 'per_fold_metrics.csv')
print('Per-fold results:')
print(df.round(4))

summary = df[['accuracy', 'f1_macro', 'auc_macro_ovr', 'precision_macro', 'recall_macro']].agg(['mean', 'std'])
print('\nSummary (mean +/- std):')
print(summary.round(4))

## ROC curves (averaged across folds)

In [ ]:
CLASS_NAMES = ['Adeno', 'SCC', 'SCLC', 'Normal']
COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e']

all_y_true, all_probs = [], []
for fold_dir in sorted(RUN_DIR.glob('fold*')):
    metrics_file = fold_dir / 'metrics.json'
    if not metrics_file.exists():
        continue
    metrics = json.loads(metrics_file.read_text())
    # NOTE: probs and y_true are stored as lists in metrics.json
    if 'probs' in metrics and 'y_true' in metrics:
        all_probs.append(np.array(metrics['probs']))
        all_y_true.append(np.array(metrics['y_true']))

if all_probs:
    probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_y_true)
    plt.figure(figsize=(7.5, 6))
    for c, name, col in zip(range(4), CLASS_NAMES, COLORS):
        yb = (y_true == c).astype(int)
        fpr, tpr, _ = roc_curve(yb, probs[:, c])
        plt.plot(fpr, tpr, lw=2.5, color=col, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves (5-fold aggregated)')
    plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Per-class clinical metrics

In [ ]:
from src.utils.metrics import per_class_clinical_metrics

if all_probs:
    pred = probs.argmax(axis=1)
    clinical = per_class_clinical_metrics(y_true, pred, n_classes=4, class_names=CLASS_NAMES)
    print(clinical.round(4))